<a href="https://colab.research.google.com/github/Manvit07/health_qa_project/blob/main/phase_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd, re

# --- Load your actual files ---
df_mp = pd.read_csv("medlineplus_corpus_curated.csv")
df_wiki = pd.read_csv("wikipedia_corpus_curated.csv")
df_who = pd.read_csv("who_corpus_raw.csv")
df_who = df_who[df_who["title"] != "Breastfeeding"].reset_index(drop=True)  # drop the soft-404

# --- Standardize schema ---
df_mp["source"] = "MedlinePlus"
df_mp["attribution"] = df_mp.apply(lambda r: f'Adapted from MedlinePlus (NIH), "{r["title"]}", {r["url"]}', axis=1)
MP_HIGH_RISK = {"Child Abuse", "Elder Abuse", "Intimate Partner Violence", "Sexual Assault", "Suicide", "Self-Harm"}
df_mp["high_risk"] = df_mp["title"].isin(MP_HIGH_RISK)
df_mp = df_mp.rename(columns={"groups": "category"})[["source","title","url","category","high_risk","summary","attribution"]]

df_wiki2 = df_wiki.copy()
df_wiki2["category"] = "Wikipedia"
df_wiki2 = df_wiki2[["source","title","url","category","high_risk","summary","attribution"]]

df_who2 = df_who.copy()
df_who2["source"] = "WHO"
df_who2["category"] = "WHO fact sheet"
df_who2["attribution"] = df_who2.apply(lambda r: f'Adapted from WHO, "{r["title"]}", {r["url"]} (CC BY-NC-SA 3.0 IGO)', axis=1)
df_who2 = df_who2[["source","title","url","category","high_risk","summary","attribution"]]

corpus = pd.concat([df_mp, df_wiki2, df_who2], ignore_index=True)
corpus["doc_id"] = range(len(corpus))
print(f"Merged corpus: {len(corpus)} documents")

Merged corpus: 319 documents


In [2]:
# --- Chunking ---
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(text)) if s.strip()]

def chunk_text(text, max_words=160, overlap_sentences=1):
    sentences = split_sentences(text)
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        wc = len(sent.split())
        if current_len + wc > max_words and current:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []
            current_len = sum(len(s.split()) for s in current)
        current.append(sent)
        current_len += wc
    if current:
        chunks.append(" ".join(current))
    return chunks

# --- Three tagging axes ---
POLICY_KEYWORDS = ["who response", "member state", "prevalence", "burden of disease", "dalys",
                    "surveillance", "sustainable development goal", "global target", "epidemiolog",
                    "financial cost", "economic cost", "cost of physical inactivity"]

# Clinical-depth: protocol acronyms, named drugs/drug classes, biochem/pharm jargon, procedural verbs
PROTOCOL_ACRONYMS = [r"\bABCDE\b", r"\bATLS\b", r"\bBATLS\b", r"\bCABD\b", r"\bAMEGA\b",
                     r"\bSAFE-POINT\b", r"\bGCS\b", r"\bAVPU\b", r"\bSOAP\b", r"\bACLS\b",
                     r"\bBLS\b", r"\bPALS\b"]
CLINICAL_VOCAB = ["cytokine", "antibody", "antibodies", "receptor", "pathway", "degranulation",
                  "bronchoconstriction", "vasodilation", "mechanism of action", "enzyme",
                  "differential diagnosis", "contraindicat", "titrat", "auscultat", "palpat",
                  "intubat", "mg/kg", "intravenous", "intramuscular", "subcutaneous",
                  "corticosteroid", "prednisone", "albuterol", "ipratropium", "fluticasone",
                  "zafirlukast", "beta-adrenergic", "leukotriene"]

def classify_policy(text):
    hits = sum(1 for kw in POLICY_KEYWORDS if kw in text.lower())
    return "policy" if hits >= 2 else "practical"

def classify_clinical_depth(text):
    t = text.lower()
    acronym_hits = sum(1 for pat in PROTOCOL_ACRONYMS if re.search(pat, text))
    vocab_hits = sum(1 for kw in CLINICAL_VOCAB if kw in t)
    total = acronym_hits + vocab_hits
    matched = [pat.strip(r"\b") for pat in PROTOCOL_ACRONYMS if re.search(pat, text)] + \
              [kw for kw in CLINICAL_VOCAB if kw in t]
    return total, matched

chunk_records = []
for _, row in corpus.iterrows():
    for i, chunk in enumerate(chunk_text(row["summary"])):
        clin_score, clin_matches = classify_clinical_depth(chunk)
        chunk_records.append({
            "chunk_id": f"{row['doc_id']}_{i}", "doc_id": row["doc_id"],
            "title": row["title"], "source": row["source"], "category": row["category"],
            "high_risk": row["high_risk"], "attribution": row["attribution"], "url": row["url"],
            "text": chunk,
            "policy_tag": classify_policy(chunk),
            "clinical_depth_score": clin_score,
            "clinical_depth_matches": ", ".join(clin_matches),
        })

df_chunks = pd.DataFrame(chunk_records)
df_chunks["flagged_for_review"] = df_chunks["clinical_depth_score"] >= 2
print(f"{len(corpus)} documents -> {len(df_chunks)} chunks")
print(f"{df_chunks['flagged_for_review'].sum()} chunks flagged for manual review ({df_chunks['flagged_for_review'].mean():.1%})")
print(df_chunks.groupby("source")["flagged_for_review"].sum())
df_chunks.to_csv("corpus_chunks_full.csv", index=False)

# Export just the flagged ones, sorted worst-first, with a blank column for your manual verdict
review_cols = ["chunk_id", "title", "source", "clinical_depth_score", "clinical_depth_matches", "text"]
df_review = df_chunks[df_chunks["flagged_for_review"]][review_cols].sort_values("clinical_depth_score", ascending=False)
df_review["manual_verdict"] = ""  # fill in: "keep" / "exclude_from_finetune" / "trim"
df_review.to_csv("clinical_depth_review.csv", index=False)
print(f"\nExported {len(df_review)} rows to clinical_depth_review.csv for manual review")

319 documents -> 1417 chunks
25 chunks flagged for manual review (1.8%)
source
MedlinePlus     1
WHO             0
Wikipedia      24
Name: flagged_for_review, dtype: int64

Exported 25 rows to clinical_depth_review.csv for manual review


In [3]:
# 1. Drop Asthma trigger entirely
corpus = corpus[corpus["title"] != "Asthma trigger"].reset_index(drop=True)
corpus["doc_id"] = range(len(corpus))  # reindex

# 2. Re-chunk First aid excluding "Setting the priorities" section
FIRST_AID_DROP_SECTION_START = "Setting the priorities"
FIRST_AID_DROP_SECTION_END = "Key basic skills"  # exclusive — resume keeping from here

def drop_section(text, start_marker, end_marker):
    s = text.find(start_marker)
    e = text.find(end_marker)
    if s == -1 or e == -1 or e <= s:
        return text  # markers not found as expected — don't silently corrupt the text
    return text[:s] + text[e:]

fa_idx = corpus.index[corpus["title"] == "First aid"][0]
original = corpus.at[fa_idx, "summary"]
trimmed = drop_section(original, FIRST_AID_DROP_SECTION_START, FIRST_AID_DROP_SECTION_END)
assert len(trimmed) < len(original), "Section drop did not change First aid text — check markers"
corpus.at[fa_idx, "summary"] = trimmed
print(f"First aid: {len(original.split())}w -> {len(trimmed.split())}w")

# 3. Re-run chunking + tagging on the updated corpus (same functions as before)
chunk_records = []
for _, row in corpus.iterrows():
    for i, chunk in enumerate(chunk_text(row["summary"])):
        clin_score, clin_matches = classify_clinical_depth(chunk)
        chunk_records.append({
            "chunk_id": f"{row['doc_id']}_{i}", "doc_id": row["doc_id"],
            "title": row["title"], "source": row["source"], "category": row["category"],
            "high_risk": row["high_risk"], "attribution": row["attribution"], "url": row["url"],
            "text": chunk, "policy_tag": classify_policy(chunk),
            "clinical_depth_score": clin_score, "clinical_depth_matches": ", ".join(clin_matches),
        })
df_chunks = pd.DataFrame(chunk_records)

# 4. Manual exclusions from the review (single chunks, not whole docs)
MANUAL_EXCLUDE_TEXT_SNIPPETS = [
    "This is inaccurate. Some simple carbohydrates (e.g., fructose)",  # Human nutrition fructolysis tangent
]
df_chunks["exclude_from_finetune"] = df_chunks["high_risk"] | df_chunks["text"].apply(
    lambda t: any(snip in t for snip in MANUAL_EXCLUDE_TEXT_SNIPPETS))

df_chunks.to_csv("corpus_chunks_final.csv", index=False)
print(f"\nFinal corpus: {corpus['doc_id'].nunique()} documents -> {len(df_chunks)} chunks")
print(f"{df_chunks['exclude_from_finetune'].sum()} chunks excluded from instruction-pair generation (high-risk + manual)")

First aid: 2822w -> 1822w

Final corpus: 318 documents -> 1393 chunks
89 chunks excluded from instruction-pair generation (high-risk + manual)
